In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass

import math

torch.manual_seed(2006)

In [2]:
@dataclass
class GPTConfig:
    block_size: int = 512 # 就是seq_len（max_seq_len）
    batch_size: int = 12
    n_layer: int = 6
    n_head: int = 12
    n_embed: int = 768 # 默认了hidden_dim和这个一个维度了
    head_size: int = n_embed // n_head
    dropout: float = 0.1
    # tiktoken 使用的是GPT-2的词表，大约有50257个token
    vocab_size: int = 50257


##### 1.register_buffer

*好处*:
- 不会随着模型训练更新(与 nn.Parameter() 区别，有参数presistent控制)
- 能够随着 model.to("cuda") 自动切换到GPU
- 能够直接跟着保存权重 state_dict


##### 2.masked_fill

- tensor.masked_fill(mask, value)

其中，mask:掩码矩阵，值为1或者True的直接被覆盖;value:覆盖在选定位置的值，如:-inf

一般与 torch.tril(torch.ones(size=(some_matrix_shape))) 搭配使用

In [3]:
class SingleHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        # 实际上后面的n_embed就是hidden_dim，只是这里正好相等
        self.q = nn.Linear(config.n_embed, config.head_size)
        self.k = nn.Linear(config.n_embed, config.head_size)
        self.v = nn.Linear(config.n_embed, config.head_size)

        # tril: 以“l”结尾，就是“lower”，i.e. 1的下三角矩阵
        # triu: 以“u”结尾，就是“upper”，i.e. 1的上三角矩阵
        self.register_buffer(
            "attention_mask",
            torch.tril(
                torch.ones(config.block_size, config.block_size)
            )
        )

        self.dropout = nn.Dropout(config.dropout)
        # self.softmax = nn.Softmax()

    attention_mask: torch.Tensor

    def forward(self, x):
        batch_size, seq_len, hidden_size = x.size()

        # [batch_size, seq_len, head_size]
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        # [batch_size, seq_len, seq_len]
        attn = Q @ K.transpose(-2, -1)

        # 经过 == 0 变成了1的上三角矩阵，填入 float("-inf")
        attn = attn.masked_fill(
            self.attention_mask[:seq_len, :seq_len] == 0,
            float('-inf')
        ) / math.sqrt(hidden_size)

        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        # [batch_size, seq_len, head_size]
        x = attn @ V

        return x

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.proj = nn.Linear(config.n_embed, config.n_embed)
        self.nets = nn.ModuleList([
            SingleHeadAttention(config) for _ in range(config.n_head)
        ])
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # [batch_size, seq_len, n_embed] 
        x = torch.cat(
            [h(x) for h in self.nets],
            dim = -1
        )

        x = self.proj(x)
        x = self.dropout(x)

        return x